# 05 — Statistical Analysis

## Purpose

Use statistical tests to formally evaluate relationships identified during EDA.

### Analytical workflow

**Research question → hypothesis → test → p-value/effect → interpretation → business meaning**

The analysis uses the same ** processed datasets** used by SQL and EDA.

> The data is synthetic and represents a simulated Zimbabwean trucking operation.

### Important

A statistically significant result does not automatically mean a relationship is practically important or causal. We will report the effect size and context where appropriate.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

PROJECT = Path.cwd().parent
DATA_DIR = PROJECT / "data" / "processed"

trips = pd.read_csv(DATA_DIR / "trips_clean.csv")
trucks = pd.read_csv(DATA_DIR / "trucks_clean.csv")
maintenance = pd.read_csv(DATA_DIR / "maintenance_clean.csv")

print("Trips:", trips.shape)
print("Trucks:", trucks.shape)
print("Maintenance:", maintenance.shape)


## 1. Pearson correlation — distance and fuel consumption

**Research question:** Is trip distance associated with fuel consumed?

### Hypotheses

- **H₀:** There is no linear association between distance and fuel consumed.
- **H₁:** There is a linear association between distance and fuel consumed.

Pearson's correlation is appropriate here because both variables are continuous and the question concerns a linear association.


In [ ]:
r, p = stats.pearsonr(
    trips["distance_km"],
    trips["fuel_consumed_litres"]
)

print(f"Pearson r = {r:.3f}")
print(f"p-value = {p:.6g}")


**Interpretation:** A large absolute value of `r` indicates a strong linear association. A small p-value provides evidence against the null hypothesis. This does not establish causation.


## 2. Pearson correlation — cargo weight and fuel consumption

**Research question:** Is cargo weight associated with fuel consumed?

This test is useful because cargo load is a plausible operational factor, but EDA should determine whether the relationship is actually present in the data.


In [ ]:
r, p = stats.pearsonr(
    trips["cargo_weight_tons"],
    trips["fuel_consumed_litres"]
)

print(f"Pearson r = {r:.3f}")
print(f"p-value = {p:.6g}")


**Interpretation:** Use both the correlation coefficient and p-value. A weak/non-significant result should be reported honestly rather than forced into a business conclusion.


## 3. Pearson correlation — truck age and maintenance frequency

**Research question:** Are older trucks associated with more maintenance events?

### Hypotheses

- **H₀:** Truck age and maintenance count are not linearly associated.
- **H₁:** Truck age and maintenance count are linearly associated.


In [ ]:
r, p = stats.pearsonr(
    trucks["age"],
    trucks["maintenance_count"]
)

print(f"Pearson r = {r:.3f}")
print(f"p-value = {p:.6g}")


**Business interpretation:** If the association is strong and statistically significant, it supports investigating age-based preventive-maintenance planning. It is still an association, not proof that age alone causes maintenance events.


## 4. Pearson correlation — mileage and maintenance frequency

**Research question:** Are higher-mileage trucks associated with more maintenance events?


In [ ]:
r, p = stats.pearsonr(
    trucks["mileage_km"],
    trucks["maintenance_count"]
)

print(f"Pearson r = {r:.3f}")
print(f"p-value = {p:.6g}")


**Interpretation:** This test evaluates whether accumulated mileage is associated with maintenance frequency across the simulated fleet.


## 5. Independent-samples t-test — profit and delivery delays

**Research question:** Do trips with delays have different average profit from trips without delays?

### Hypotheses

- **H₀:** Mean profit is the same for delayed and non-delayed trips.
- **H₁:** Mean profit differs between delayed and non-delayed trips.

Welch's t-test is used because it does not assume equal variances between the two groups.


In [ ]:
delayed_profit = trips.loc[trips["has_delay"] == 1, "profit"]
on_time_profit = trips.loc[trips["has_delay"] == 0, "profit"]

t_stat, p = stats.ttest_ind(
    delayed_profit,
    on_time_profit,
    equal_var=False
)

mean_delayed = delayed_profit.mean()
mean_on_time = on_time_profit.mean()

print(f"Mean profit — delayed: ${mean_delayed:,.2f}")
print(f"Mean profit — no delay: ${mean_on_time:,.2f}")
print(f"Welch t-statistic = {t_stat:.3f}")
print(f"p-value = {p:.6g}")


**Interpretation:** A statistically significant difference indicates that the average profit differs between the two groups. It does not by itself identify why the difference exists.


## 6. Chi-square test — border crossing and delays

**Research question:** Is delay status associated with whether a trip involves a border crossing?

### Hypotheses

- **H₀:** Border-crossing status and delay status are independent.
- **H₁:** They are associated.

Chi-square is appropriate because both variables are categorical.


In [ ]:
contingency = pd.crosstab(
    trips["border_crossing_flag"],
    trips["has_delay"]
)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print("Contingency table:")
display(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p:.6g}")


**Business interpretation:** If the association is statistically significant, border-crossing operations deserve further operational investigation when planning delivery schedules. The test does not establish that border crossings alone cause delays.


## 7. Chi-square test — road condition and delays

**Research question:** Is delay status associated with road condition?


In [ ]:
contingency = pd.crosstab(
    trips["road_condition"],
    trips["has_delay"]
)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print("Contingency table:")
display(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p:.6g}")


**Interpretation:** A significant association indicates that delay frequency differs across road-condition categories. This supports further investigation of road conditions as an operational risk factor.


## 8. Chi-square test — weather condition and delays

**Research question:** Is delay status associated with weather condition?


In [ ]:
contingency = pd.crosstab(
    trips["weather_condition"],
    trips["has_delay"]
)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print("Contingency table:")
display(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p:.6g}")


**Interpretation:** This identifies whether delay frequency differs across weather categories in the simulated data.


## 9. One-way ANOVA — route and trip profit

**Research question:** Does average trip profit differ across routes?

### Hypotheses

- **H₀:** All route groups have the same mean profit.
- **H₁:** At least one route has a different mean profit.

ANOVA tests the overall difference. If significant, a post-hoc analysis would be needed to identify which specific routes differ.


In [ ]:
route_groups = [
    group["profit"].values
    for _, group in trips.groupby("route")
    if len(group) >= 2
]

f_stat, p = stats.f_oneway(*route_groups)

print(f"ANOVA F-statistic = {f_stat:.3f}")
print(f"p-value = {p:.6g}")


**Interpretation:** A significant ANOVA result indicates that average profitability is not the same across all route groups. It does not identify the specific route differences by itself.


## 10. Effect-size and practical interpretation

Statistical significance can be influenced by sample size. For the t-test, calculate Cohen's *d* to describe the standardized difference between delayed and non-delayed trip profit.


In [ ]:
# Cohen's d for delayed vs non-delayed trip profit.

n1, n2 = len(delayed_profit), len(on_time_profit)
s1, s2 = delayed_profit.std(ddof=1), on_time_profit.std(ddof=1)

pooled_sd = np.sqrt(
    ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) /
    (n1 + n2 - 2)
)

cohens_d = (mean_delayed - mean_on_time) / pooled_sd

print(f"Cohen's d = {cohens_d:.3f}")


**Interpretation:** Cohen's *d* expresses the difference between the two group means in standard-deviation units. Use it alongside the p-value and the actual dollar difference.


## 11. Multiple-testing caution

This notebook evaluates several hypotheses. When many statistical tests are performed, the chance of obtaining at least one small p-value by chance increases.

For the final report:

- treat p-values as evidence rather than absolute truth;
- emphasize effect sizes and business relevance;
- avoid claiming causation;
- clearly state that the dataset is synthetic;
- avoid selecting only significant results.

The purpose of this phase is to validate important EDA findings, not to manufacture significance.


## 12. Automatic statistical findings summary

The table below is generated automatically from the statistical tests above.

This prevents manual copying of statistics and keeps the final summary reproducible.

### Evidence rule

- `p < 0.05` → statistically significant at the 5% level.
- `p >= 0.05` → not statistically significant at the 5% level.

Statistical significance is reported separately from practical/business importance.


In [ ]:
ALPHA = 0.05
results = []

def evidence(p):
    return "Statistically significant at 5%" if p < ALPHA else "Not statistically significant at 5%"

# 1. Distance vs fuel consumption
r, p = stats.pearsonr(trips["distance_km"], trips["fuel_consumed_litres"])
results.append([
    "Distance vs fuel consumption", "Pearson correlation",
    f"r = {r:.3f}", p, evidence(p),
    "Distance and fuel consumed show a linear association in the simulated trips."
])

# 2. Cargo weight vs fuel consumption
r, p = stats.pearsonr(trips["cargo_weight_tons"], trips["fuel_consumed_litres"])
results.append([
    "Cargo weight vs fuel consumption", "Pearson correlation",
    f"r = {r:.3f}", p, evidence(p),
    "Cargo weight and fuel consumed show a linear association in the simulated trips."
])

# 3. Truck age vs maintenance count
r, p = stats.pearsonr(trucks["age"], trucks["maintenance_count"])
results.append([
    "Truck age vs maintenance count", "Pearson correlation",
    f"r = {r:.3f}", p, evidence(p),
    "Truck age and maintenance frequency are associated in the simulated fleet."
])

# 4. Mileage vs maintenance count
r, p = stats.pearsonr(trucks["mileage_km"], trucks["maintenance_count"])
results.append([
    "Mileage vs maintenance count", "Pearson correlation",
    f"r = {r:.3f}", p, evidence(p),
    "Truck mileage and maintenance frequency are associated in the simulated fleet."
])

# 5. Delay vs profit
delayed_profit = trips.loc[trips["has_delay"] == 1, "profit"]
on_time_profit = trips.loc[trips["has_delay"] == 0, "profit"]
t_stat, p = stats.ttest_ind(delayed_profit, on_time_profit, equal_var=False)
results.append([
    "Delay vs profit", "Welch t-test",
    f"t = {t_stat:.3f}", p, evidence(p),
    "Average trip profit differs between delayed and non-delayed trips."
    if p < ALPHA else
    "There is not sufficient evidence of a difference in average profit between delayed and non-delayed trips."
])

# 6. Border crossing vs delay
table = pd.crosstab(trips["border_crossing_flag"], trips["has_delay"])
chi2, p, dof, expected = stats.chi2_contingency(table)
results.append([
    "Border crossing vs delay", "Chi-square",
    f"χ² = {chi2:.3f}", p, evidence(p),
    "Border-crossing status and delay status are associated."
    if p < ALPHA else
    "There is not sufficient evidence of an association between border-crossing status and delay status."
])

# 7. Road condition vs delay
table = pd.crosstab(trips["road_condition"], trips["has_delay"])
chi2, p, dof, expected = stats.chi2_contingency(table)
results.append([
    "Road condition vs delay", "Chi-square",
    f"χ² = {chi2:.3f}", p, evidence(p),
    "Road-condition category and delay status are associated."
    if p < ALPHA else
    "There is not sufficient evidence of an association between road condition and delay status."
])

# 8. Weather vs delay
table = pd.crosstab(trips["weather_condition"], trips["has_delay"])
chi2, p, dof, expected = stats.chi2_contingency(table)
results.append([
    "Weather vs delay", "Chi-square",
    f"χ² = {chi2:.3f}", p, evidence(p),
    "Weather category and delay status are associated."
    if p < ALPHA else
    "There is not sufficient evidence of an association between weather and delay status."
])

# 9. Route vs profit
route_groups = [
    group["profit"].values
    for _, group in trips.groupby("route")
    if len(group) >= 2
]
f_stat, p = stats.f_oneway(*route_groups)
results.append([
    "Route vs profit", "One-way ANOVA",
    f"F = {f_stat:.3f}", p, evidence(p),
    "Average trip profit differs across at least some route groups."
    if p < ALPHA else
    "There is not sufficient evidence that average trip profit differs across route groups."
])

results_df = pd.DataFrame(
    results,
    columns=[
        "Research question", "Test", "Statistic",
        "p-value", "Evidence", "Business interpretation"
    ]
)

results_df["p-value"] = results_df["p-value"].map(lambda x: f"{x:.6g}")
display(results_df)


### How to read the automatic summary

- **Pearson `r`** describes the direction and strength of a linear association.
- **Welch `t`** compares the means of two groups without assuming equal variances.
- **Chi-square `χ²`** tests association between categorical variables.
- **ANOVA `F`** tests whether mean profit differs across multiple route groups.
- **p-value** is compared with the 0.05 threshold used in this notebook.

The wording deliberately describes **association or difference**, not causation.


## 13. Effect size — delayed vs non-delayed profit

A p-value tells us whether the observed difference is statistically detectable. Cohen's *d* describes the standardized size of the difference.


In [ ]:
delayed_profit = trips.loc[trips["has_delay"] == 1, "profit"]
on_time_profit = trips.loc[trips["has_delay"] == 0, "profit"]

n1, n2 = len(delayed_profit), len(on_time_profit)
s1, s2 = delayed_profit.std(ddof=1), on_time_profit.std(ddof=1)

pooled_sd = np.sqrt(
    ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) /
    (n1 + n2 - 2)
)

cohens_d = (delayed_profit.mean() - on_time_profit.mean()) / pooled_sd

print(f"Delayed-trip mean profit: ${delayed_profit.mean():,.2f}")
print(f"Non-delayed mean profit: ${on_time_profit.mean():,.2f}")
print(f"Mean difference: ${(delayed_profit.mean() - on_time_profit.mean()):,.2f}")
print(f"Cohen's d: {cohens_d:.3f}")


## 14. Multiple-testing caution

Several hypotheses are tested in this phase. Therefore, individual p-values should not be treated as standalone proof.

For the final report:

- report the planned tests;
- show the actual statistics and p-values;
- consider effect size and business relevance;
- avoid causal claims;
- state that the dataset is synthetic;
- do not select only significant findings.

### Final learning points

You should now be able to explain:

1. When to use Pearson correlation.
2. How a Welch t-test compares two group means.
3. How chi-square tests categorical association.
4. How ANOVA tests differences across multiple groups.
5. Why p-values and effect sizes should be interpreted together.
6. Why statistical association is not the same as causation.
7. How to create a reproducible statistical-results summary.

### Next phase

**06 — Machine Learning **
